In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count, month, year

# Create Spark Session
spark = SparkSession.builder \
    .appName("EcommerceTransformation") \
    .getOrCreate()

print("Spark Session Created Successfully! ✅")

Spark Session Created Successfully! ✅


In [ ]:
# Load all CSV files
orders = spark.read.csv("olist_orders_dataset.csv", header=True, inferSchema=True)
customers = spark.read.csv("olist_customers_dataset.csv", header=True, inferSchema=True)
order_items = spark.read.csv("olist_order_items_dataset.csv", header=True, inferSchema=True)
payments = spark.read.csv("olist_order_payments_dataset.csv", header=True, inferSchema=True)
products = spark.read.csv("olist_products_dataset.csv", header=True, inferSchema=True)
reviews = spark.read.csv("olist_order_reviews_dataset.csv", header=True, inferSchema=True)

print("All files loaded successfully! ✅")

All files loaded successfully! ✅


In [ ]:
# Clean Orders table
orders_clean = orders.dropna(subset=["order_id", "customer_id"]) \
    .dropDuplicates(["order_id"])

# Clean Customers table
customers_clean = customers.dropna(subset=["customer_id"]) \
    .dropDuplicates(["customer_id"])

# Clean Order Items table
order_items_clean = order_items.dropna(subset=["order_id", "product_id"]) \
    .dropDuplicates()

# Clean Payments table
payments_clean = payments.dropna(subset=["order_id"]) \
    .dropDuplicates()

# Clean Products table
products_clean = products.dropna(subset=["product_id"]) \
    .dropDuplicates(["product_id"])

# Clean Reviews table
reviews_clean = reviews.dropna(subset=["order_id"]) \
    .dropDuplicates()

print("Data Cleaned Successfully! ✅")

Data Cleaned Successfully! ✅


In [ ]:
# Join Orders with Customers
orders_customers = orders_clean.join(customers_clean,
                    on="customer_id",
                    how="left")

# Join with Order Items
orders_items = orders_customers.join(order_items_clean,
                    on="order_id",
                    how="left")

# Join with Payments
orders_payments = orders_items.join(payments_clean,
                    on="order_id",
                    how="left")

# Join with Products
final_df = orders_payments.join(products_clean,
                    on="product_id",
                    how="left")

print("Tables Joined Successfully! ✅")
print("Total Records:", final_df.count())

Tables Joined Successfully! ✅
Total Records: 118434


In [ ]:
# Save final clean data as CSV
final_df.write.csv("ecommerce_clean_data",
                    header=True,
                    mode="overwrite")

print("Clean Data Saved Successfully! ✅")

Clean Data Saved Successfully! ✅


In [ ]:
# Convert to single CSV file and download
final_df.toPandas().to_csv("ecommerce_final_clean.csv", index=False)

print("File Ready to Download! ✅")

File Ready to Download! ✅


In [ ]:
from google.colab import files
files.download("ecommerce_final_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

df = pd.read_csv("ecommerce_final_clean.csv")
print(df.columns.tolist())
print("Total columns:", len(df.columns))

['product_id', 'order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
Total columns: 30
